# LeetCode 852 — Peak Index in a Mountain Array bằng Python

Link đề: https://leetcode.com/problems/peak-index-in-a-mountain-array/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search (tìm kiếm nhị phân), Ternary Search (tìm kiếm tam phân)

Skeleton LeetCode:

```python
class Solution:
    def peakIndexInMountainArray(self, arr: List[int]) -> int:
        ...
```

Ghi chú: mảng "ngọn núi" **tăng ngặt** tới đỉnh rồi **giảm ngặt**; cần chỉ số của đỉnh trong `O(log n)`. Điểm mấu chốt: dù mảng **không được sắp xếp**, nó có tính chất **đơn điệu theo độ dốc**: ở bên trái đỉnh `arr[i] < arr[i + 1]`, từ đỉnh trở đi `arr[i] > arr[i + 1]`. Một tính chất "sai ... sai, đúng ... đúng" là đủ để **chặt nhị phân**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mảng tăng ngặt tới một đỉnh rồi giảm ngặt; trả chỉ số đỉnh; yêu cầu thời gian `O(log n)`.
2. Nắm các công cụ Python cần thiết: tìm kiếm nhị phân theo khuôn `while lo < hi`, `mid = (lo + hi) >> 1`, `bisect_left` với tham số `key` (Python 3.10+), `max` với `key`, và lớp bao để **đếm số lần đọc mảng** nhằm chứng minh độ phức tạp.
3. Hiểu vì sao chặt nhị phân đúng khi chỉ biết **tính chất đơn điệu của vị từ** `arr[i] > arr[i + 1]` (không cần mảng sắp xếp).
4. Đi qua 5 cách: quét tuyến tính, nhị phân trên độ dốc (lời giải chuẩn), nhị phân trên `[0, n-1]`, tam phân, và `bisect_left` với `key`; đối chiếu bằng oracle `argmax` và **đếm số lần đọc**.
5. Tránh các lỗi kinh điển: `while lo <= hi` kèm `hi = mid` (lặp vô hạn), `arr[mid + 1]` vượt biên, nhầm hướng thu hẹp, trả về **giá trị** thay vì **chỉ số**, và quét tuyến tính khi đề đòi `O(log n)`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `arr` độ dài `n` là **mảng ngọn núi**: các giá trị **tăng** tới một **phần tử đỉnh** rồi **giảm**.

Trả về **chỉ số** của phần tử đỉnh. Yêu cầu giải trong thời gian `O(log n)`.

**Ví dụ 1:** `arr = [0, 1, 0]` -> `1`

**Ví dụ 2:** `arr = [0, 2, 1, 0]` -> `1`

**Ví dụ 3:** `arr = [0, 10, 5, 2]` -> `1`

**Ràng buộc:**

- `3 <= arr.length <= 10^5`
- `0 <= arr[i] <= 10^6`
- `arr` **đảm bảo** là mảng ngọn núi (tăng **ngặt** tới đỉnh rồi giảm **ngặt**, và đỉnh không nằm ở hai đầu).

In [1]:
from typing import List
from bisect import bisect_left
import random, time
from itertools import product, combinations

examples = [
    ([0, 1, 0], 1),
    ([0, 2, 1, 0], 1),
    ([0, 10, 5, 2], 1),
]
for arr, expected in examples:
    print(f"arr={arr} kỳ vọng={expected}")

arr=[0, 1, 0] kỳ vọng=1
arr=[0, 2, 1, 0] kỳ vọng=1
arr=[0, 10, 5, 2] kỳ vọng=1


## 3. Kiến thức nền

### 3.1. Mảng ngọn núi

`arr` là ngọn núi khi tồn tại chỉ số đỉnh `p` với `0 < p < n - 1`, `arr[0] < arr[1] < ... < arr[p]` và `arr[p] > arr[p + 1] > ... > arr[n - 1]`. Hai hệ quả:

- Đỉnh **duy nhất** và là phần tử **lớn nhất** của mảng.
- Với mọi `i` trong `[0, n - 2]`: `arr[i] < arr[i + 1]` **khi và chỉ khi** `i < p`; còn `arr[i] > arr[i + 1]` **khi và chỉ khi** `i >= p`.

In [2]:
def is_mountain(arr) -> bool:
    n = len(arr)
    if n < 3:
        return False
    i = 0
    while i + 1 < n and arr[i] < arr[i + 1]:
        i += 1
    if i == 0 or i == n - 1:
        return False
    while i + 1 < n and arr[i] > arr[i + 1]:
        i += 1
    return i == n - 1

for arr in ([0, 2, 1, 0], [1, 2, 3], [3, 2, 1], [1, 3, 3, 1], [1, 2, 1, 2]):
    print(arr, "-> ngọn núi?", is_mountain(arr))

[0, 2, 1, 0] -> ngọn núi? True
[1, 2, 3] -> ngọn núi? False
[3, 2, 1] -> ngọn núi? False
[1, 3, 3, 1] -> ngọn núi? False
[1, 2, 1, 2] -> ngọn núi? False


### 3.2. Tính chất đơn điệu của vị từ độ dốc

Đặt `down(i) = (arr[i] > arr[i + 1])` cho `i = 0..n-2`. Với ngọn núi, dãy `down(0), down(1), ..., down(n-2)` có dạng `False, ..., False, True, ..., True`; **vị trí `True` đầu tiên chính là đỉnh**. Dãy "sai rồi đúng" như vậy là đối tượng của **tìm kiếm nhị phân** (không cần mảng được sắp xếp).

In [3]:
arr = [0, 2, 5, 9, 8, 3, 1]
print("i      :", list(range(len(arr) - 1)))
print("down(i):", [arr[i] > arr[i + 1] for i in range(len(arr) - 1)])
print("đỉnh = chỉ số True đầu tiên =", [arr[i] > arr[i + 1] for i in range(len(arr) - 1)].index(True))

i      : [0, 1, 2, 3, 4, 5]
down(i): [False, False, False, True, True, True]
đỉnh = chỉ số True đầu tiên = 3


### 3.3. Khuôn tìm kiếm nhị phân `while lo < hi`

Với bài "tìm vị trí đầu tiên thỏa vị từ" (sai ... sai, đúng ... đúng):

```
lo, hi = 0, n - 1              # đáp án chắc chắn nằm trong [lo, hi]
while lo < hi:
    mid = (lo + hi) >> 1       # làm tròn xuống, luôn mid < hi
    if vị_từ(mid): hi = mid    # mid có thể là đáp án
    else:          lo = mid + 1
return lo
```

Vì `mid < hi` nên `mid + 1 <= hi <= n - 1` (không vượt biên khi đọc `arr[mid + 1]`).

In [4]:
a = [1, 3, 4, 7, 9, 12]
lo, hi = 0, len(a) - 1
while lo < hi:
    mid = (lo + hi) >> 1
    print(f"lo={lo} hi={hi} mid={mid} a[mid]={a[mid]}", "-> hi = mid" if a[mid] >= 7 else "-> lo = mid + 1")
    if a[mid] >= 7:
        hi = mid
    else:
        lo = mid + 1
print("vị trí đầu tiên có giá trị >= 7:", lo)

lo=0 hi=5 mid=2 a[mid]=4 -> lo = mid + 1
lo=3 hi=5 mid=4 a[mid]=9 -> hi = mid
lo=3 hi=4 mid=3 a[mid]=7 -> hi = mid
vị trí đầu tiên có giá trị >= 7: 3


### 3.4. `bisect_left` với `key` (Python 3.10+)

`bisect_left(a, x, key=f)` tìm vị trí đầu tiên mà `f(a[i]) >= x`, giả định dãy `f(a[i])` không giảm. Với `a = range(n - 1)` và `f = lambda i: arr[i] > arr[i + 1]` (kiểu `bool`, `False < True`), `bisect_left(a, True, key=f)` cho vị trí `True` đầu tiên.

In [5]:
arr = [0, 2, 5, 9, 8, 3, 1]
peak = bisect_left(range(len(arr) - 1), True, key=lambda i: arr[i] > arr[i + 1])
print("đỉnh:", peak, "| arr[đỉnh] =", arr[peak])

đỉnh: 3 | arr[đỉnh] = 9


### 3.5. Đếm số lần đọc mảng

Để **chứng minh** độ phức tạp ta bọc mảng trong một lớp đếm số lần `arr[i]` được đọc. Quét tuyến tính đọc `O(n)` lần, nhị phân chỉ `O(log n)`.

In [6]:
class Counted:
    """Bao một list và đếm số lần đọc phần tử."""
    def __init__(self, data):
        self.data = data
        self.reads = 0
    def __getitem__(self, i):
        self.reads += 1
        return self.data[i]
    def __len__(self):
        return len(self.data)

c = Counted([0, 2, 5, 9, 8, 3, 1])
_ = c[3], c[4], c[3]
print("số lần đọc:", c.reads, "| độ dài:", len(c))

số lần đọc: 3 | độ dài: 7


## 4. Ý tưởng nền tảng / trực giác (tính tay)

`arr = [0, 2, 5, 9, 8, 3, 1]` (đỉnh ở chỉ số 3). Thử phỏng đoán đỉnh bằng cách nhìn **một cặp kề nhau**:

- Nếu đứng ở `i` mà `arr[i] < arr[i + 1]` (đang **đi lên**) thì đỉnh nằm **bên phải** `i` (ở `i + 1` hoặc xa hơn).
- Nếu `arr[i] > arr[i + 1]` (đang **đi xuống**) thì đỉnh nằm **ở `i` hoặc bên trái**.

Mỗi lần xem một cặp ta **loại nửa** vùng tìm: đó là chặt nhị phân.

Tính tay: `lo = 0`, `hi = 6`; `mid = 3`: `arr[3] = 9 > arr[4] = 8` (đang xuống) -> `hi = 3`; `mid = 1`: `arr[1] = 2 < arr[2] = 5` (đang lên) -> `lo = 2`; `mid = 2`: `5 < 9` -> `lo = 3`; `lo == hi = 3`: đỉnh ở 3.

In [7]:
def peak_hand(arr):
    lo, hi = 0, len(arr) - 1
    while lo < hi:
        mid = (lo + hi) >> 1
        up = arr[mid] < arr[mid + 1]
        print(f"lo={lo} hi={hi} mid={mid}: arr[{mid}]={arr[mid]} {'<' if up else '>'} arr[{mid + 1}]={arr[mid + 1]} -> {'đang lên: lo = mid + 1' if up else 'đang xuống: hi = mid'}")
        if up:
            lo = mid + 1
        else:
            hi = mid
    return lo

print("đỉnh =", peak_hand([0, 2, 5, 9, 8, 3, 1]))

lo=0 hi=6 mid=3: arr[3]=9 > arr[4]=8 -> đang xuống: hi = mid
lo=0 hi=3 mid=1: arr[1]=2 < arr[2]=5 -> đang lên: lo = mid + 1
lo=2 hi=3 mid=2: arr[2]=5 < arr[3]=9 -> đang lên: lo = mid + 1
đỉnh = 3


## 5. Cách 1 — Brute Force: quét tuyến tính

**Ý tưởng.** Duyệt từ trái sang phải, dừng ở chỉ số đầu tiên mà `arr[i] > arr[i + 1]` (hoặc đơn giản: `arr.index(max(arr))`). Đây là cách hiển nhiên nhưng `O(n)`. Nó cũng là **oracle** (đỉnh là phần tử lớn nhất duy nhất).

In [8]:
def peak_linear(arr: List[int]) -> int:
    for i in range(len(arr) - 1):
        if arr[i] > arr[i + 1]:
            return i
    return len(arr) - 1

def peak_argmax(arr: List[int]) -> int:
    return arr.index(max(arr))

for arr, expected in examples:
    assert peak_linear(arr) == peak_argmax(arr) == expected
print("quét tuyến tính OK; argmax khớp")

quét tuyến tính OK; argmax khớp


**Độ phức tạp và nút thắt.** `O(n)` thời gian, `O(1)` bộ nhớ. Đề **yêu cầu `O(log n)`**: với `n = 10^5` quét tuyến tính đọc tới `10^5` phần tử. Hãy đếm số lần đọc:

In [9]:
def mountain(n, peak, rng=None):
    """Mảng ngọn núi độ dài n, đỉnh ở chỉ số peak (0 < peak < n - 1), giá trị trong [0, 10^6]."""
    rng = rng or random.Random(0)
    values = rng.sample(range(0, 10 ** 6 + 1), n)
    values.sort()
    top = values.pop()                                # giá trị lớn nhất là đỉnh
    left_vals = rng.sample(values, peak)              # chọn peak giá trị cho sườn trái
    left = sorted(left_vals)
    left_set = set(left_vals)
    rest = [v for v in values if v not in left_set]
    right = sorted(rest, reverse=True)
    return left + [top] + right

for n in (100, 1000, 10000, 100000):
    arr = mountain(n, n // 3, random.Random(n))
    assert is_mountain(arr) and len(arr) == n
    c = Counted(arr)
    peak_linear(c)
    print(f"n={n:6}: quét tuyến tính đọc {c.reads:7} lần")

n=   100: quét tuyến tính đọc      68 lần
n=  1000: quét tuyến tính đọc     668 lần
n= 10000: quét tuyến tính đọc    6668 lần


n=100000: quét tuyến tính đọc   66668 lần


Số lần đọc tăng **tuyến tính** theo `n`. Ta cần một cách đọc chỉ `O(log n)` phần tử.

## 6. Ý tưởng thuật toán / bước đột phá

**Chặt nhị phân trên vị từ độ dốc.** Vị từ `down(i) = arr[i] > arr[i + 1]` là `False, ..., False, True, ..., True`. Mỗi lần xem `mid`:

- `down(mid)` đúng -> đỉnh `<= mid`: thu hẹp về `[lo, mid]`.
- `down(mid)` sai -> đỉnh `> mid`: thu hẹp về `[mid + 1, hi]`.

Mỗi bước loại nửa vùng, `O(log n)` bước, mỗi bước đọc hai phần tử.

Các cách triển khai khác nhau của ý tưởng "loại bỏ vùng không chứa đỉnh":

| Cách | Ý tưởng |
|---|---|
| 2 | Nhị phân trên `[1, n - 2]` (đỉnh không nằm ở hai đầu) |
| 3 | Nhị phân trên `[0, n - 1]`, so `arr[mid]` với `arr[mid + 1]` |
| 4 | **Tam phân**: so `arr[m1]` với `arr[m2]`, loại một phần ba |
| 5 | `bisect_left` với `key` trên `range(n - 1)` |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Nhị phân trên `[1, n - 2]` (lời giải chuẩn)

**Giải thích từng đoạn.**

- Đỉnh không thể ở chỉ số `0` hay `n - 1` (đề đảm bảo), nên đáp án trong `[1, n - 2]`.
- `mid = (left + right) >> 1`. Nếu `arr[mid] > arr[mid + 1]`: đang xuống, đỉnh `<= mid`: `right = mid`. Ngược lại: `left = mid + 1`.
- `left == right` là đỉnh.

In [10]:
def peak_binary_inner(arr: List[int]) -> int:
    left, right = 1, len(arr) - 2
    while left < right:
        mid = (left + right) >> 1
        if arr[mid] > arr[mid + 1]:
            right = mid
        else:
            left = mid + 1
    return left

for arr, expected in examples:
    assert peak_binary_inner(arr) == expected
print("nhị phân [1, n-2] OK")

nhị phân [1, n-2] OK


**Code nộp LeetCode (Cách 2):**

In [11]:
class Solution:
    def peakIndexInMountainArray(self, arr: List[int]) -> int:
        left, right = 1, len(arr) - 2
        while left < right:
            mid = (left + right) >> 1
            if arr[mid] > arr[mid + 1]:
                right = mid
            else:
                left = mid + 1
        return left

sol_inner = Solution().peakIndexInMountainArray
print([sol_inner(a) for a, _ in examples])

[1, 1, 1]


**Độ phức tạp (Cách 2):** thời gian `O(log n)`, bộ nhớ `O(1)`.

---

### 7.2. Cách 3 — Nhị phân trên `[0, n - 1]`

**Giải thích từng đoạn.**

- `lo, hi = 0, n - 1`; trong vòng `while lo < hi` ta có `mid < hi` nên `mid + 1 <= n - 1` (an toàn).
- `arr[mid] < arr[mid + 1]` (đang lên): `lo = mid + 1`; ngược lại `hi = mid`.
- Không dựa vào việc đỉnh không ở hai đầu nên dùng được cho mảng "tăng rồi giảm" tổng quát (kể cả chỉ tăng).

In [12]:
def peak_binary_full(arr: List[int]) -> int:
    lo, hi = 0, len(arr) - 1
    while lo < hi:
        mid = (lo + hi) >> 1
        if arr[mid] < arr[mid + 1]:
            lo = mid + 1
        else:
            hi = mid
    return lo

for arr, expected in examples:
    assert peak_binary_full(arr) == expected
print("nhị phân [0, n-1] OK")

nhị phân [0, n-1] OK


**Code nộp LeetCode (Cách 3):**

In [13]:
class Solution:
    def peakIndexInMountainArray(self, arr: List[int]) -> int:
        lo, hi = 0, len(arr) - 1
        while lo < hi:
            mid = (lo + hi) >> 1
            if arr[mid] < arr[mid + 1]:
                lo = mid + 1
            else:
                hi = mid
        return lo

sol_full = Solution().peakIndexInMountainArray
print([sol_full(a) for a, _ in examples])

[1, 1, 1]


**Độ phức tạp (Cách 3):** thời gian `O(log n)`, bộ nhớ `O(1)`.

---

### 7.3. Cách 4 — Tam phân (ternary search)

**Ý tưởng khác.** Tam phân dành cho **hàm đơn đỉnh** (unimodal). Chọn hai điểm `m1 < m2` chia đoạn thành ba phần. So `arr[m1]` và `arr[m2]`:

- `arr[m1] < arr[m2]`: đỉnh **không** ở `[lo, m1]` (nếu cả hai cùng bên trái đỉnh thì `arr[m1] < arr[m2]` và đỉnh `> m1`; nếu `m2` ở bên phải đỉnh thì `arr[m1] < arr[m2]` nghĩa đỉnh vẫn `> m1`): `lo = m1 + 1`.
- `arr[m1] > arr[m2]`: đỉnh không ở `[m2, hi]`: `hi = m2 - 1`.
- `arr[m1] == arr[m2]` (chỉ xảy ra khi hai điểm ở hai phía của đỉnh vì mảng **ngặt**): đỉnh nằm giữa: `lo = m1 + 1`, `hi = m2 - 1`.

Khi còn `<= 3` phần tử, chọn max.

**Giải thích từng đoạn.** `third = (hi - lo) // 3`; `m1 = lo + third`, `m2 = hi - third`. Vòng lặp chạy khi `hi - lo > 2`; cuối cùng `max(range(lo, hi + 1), key=arr.__getitem__)`.

In [14]:
def peak_ternary(arr: List[int]) -> int:
    lo, hi = 0, len(arr) - 1
    while hi - lo > 2:
        third = (hi - lo) // 3
        m1, m2 = lo + third, hi - third
        if arr[m1] < arr[m2]:
            lo = m1 + 1
        elif arr[m1] > arr[m2]:
            hi = m2 - 1
        else:
            lo, hi = m1 + 1, m2 - 1
    return max(range(lo, hi + 1), key=lambda i: arr[i])

for arr, expected in examples:
    assert peak_ternary(arr) == expected
print("tam phân OK")

tam phân OK


**Code nộp LeetCode (Cách 4):**

In [15]:
class Solution:
    def peakIndexInMountainArray(self, arr: List[int]) -> int:
        lo, hi = 0, len(arr) - 1
        while hi - lo > 2:
            third = (hi - lo) // 3
            m1, m2 = lo + third, hi - third
            if arr[m1] < arr[m2]:
                lo = m1 + 1
            elif arr[m1] > arr[m2]:
                hi = m2 - 1
            else:
                lo, hi = m1 + 1, m2 - 1
        return max(range(lo, hi + 1), key=lambda i: arr[i])

sol_ternary = Solution().peakIndexInMountainArray
print([sol_ternary(a) for a, _ in examples])

[1, 1, 1]


**Độ phức tạp (Cách 4):** thời gian `O(log_{3/2} n)` (mỗi bước loại ít nhất một phần ba; số lần đọc nhiều hơn nhị phân vì mỗi bước đọc hai phần tử và loại `1/3` thay vì `1/2`), bộ nhớ `O(1)`. Tam phân quan trọng khi hàm không rời rạc hoặc không thể so sánh kề nhau.

---

### 7.4. Cách 5 — `bisect_left` với `key`

**Ý tưởng khác.** Tận dụng thư viện chuẩn: `bisect_left(range(n - 1), True, key=lambda i: arr[i] > arr[i + 1])` trả về chỉ số `True` đầu tiên của vị từ độ dốc (cần Python 3.10 trở lên).

**Giải thích từng đoạn.** `range(n - 1)` là tập các chỉ số `i` có `arr[i + 1]`; `key` biến chỉ số thành `bool`; `True` lớn hơn `False` nên dãy `key` không giảm (đúng yêu cầu `bisect`). Vì đỉnh luôn tồn tại trong `[1, n - 2]`, vị trí tìm được hợp lệ.

In [16]:
def peak_bisect(arr: List[int]) -> int:
    return bisect_left(range(len(arr) - 1), True, key=lambda i: arr[i] > arr[i + 1])

for arr, expected in examples:
    assert peak_bisect(arr) == expected
print("bisect_left với key OK")

bisect_left với key OK


**Code nộp LeetCode (Cách 5):** cần Python 3.10+; LeetCode dùng bản mới nên chạy được.

In [17]:
class Solution:
    def peakIndexInMountainArray(self, arr: List[int]) -> int:
        from bisect import bisect_left
        return bisect_left(range(len(arr) - 1), True, key=lambda i: arr[i] > arr[i + 1])

sol_bisect = Solution().peakIndexInMountainArray
print([sol_bisect(a) for a, _ in examples])

[1, 1, 1]


**Độ phức tạp (Cách 5):** thời gian `O(log n)`, bộ nhớ `O(1)` (`range` không tạo list).

## 8. TRACE từng bước

### 8.1. Cách 2 trên `[0, 2, 5, 9, 8, 3, 1]`

`left = 1`, `right = 5`.

| vòng | `left` | `right` | `mid` | `arr[mid]` vs `arr[mid+1]` | hành động |
|---|---|---|---|---|---|
| 1 | 1 | 5 | 3 | `9 > 8` | `right = 3` |
| 2 | 1 | 3 | 2 | `5 < 9` | `left = 3` |
| 3 | 3 | 3 | | | dừng, đỉnh = 3 |

In [18]:
def trace_inner(arr):
    left, right = 1, len(arr) - 2
    step = 0
    print("arr =", arr)
    while left < right:
        step += 1
        mid = (left + right) >> 1
        down = arr[mid] > arr[mid + 1]
        print(f"  vòng {step}: left={left} right={right} mid={mid}: arr[{mid}]={arr[mid]} {'>' if down else '<'} arr[{mid + 1}]={arr[mid + 1]} -> {'right = mid' if down else 'left = mid + 1'}")
        if down:
            right = mid
        else:
            left = mid + 1
    print(f"  dừng: left = right = {left}, arr[{left}] = {arr[left]}")
    return left

assert trace_inner([0, 2, 5, 9, 8, 3, 1]) == 3
print()
assert trace_inner([0, 10, 5, 2]) == 1

arr = [0, 2, 5, 9, 8, 3, 1]
  vòng 1: left=1 right=5 mid=3: arr[3]=9 > arr[4]=8 -> right = mid
  vòng 2: left=1 right=3 mid=2: arr[2]=5 < arr[3]=9 -> left = mid + 1
  dừng: left = right = 3, arr[3] = 9

arr = [0, 10, 5, 2]
  vòng 1: left=1 right=2 mid=1: arr[1]=10 > arr[2]=5 -> right = mid
  dừng: left = right = 1, arr[1] = 10


### 8.2. Cách 4 (tam phân) trên một mảng dài hơn

Mảng 12 phần tử `[1, 4, 7, 10, 14, 20, 19, 15, 11, 8, 3, 0]` (đỉnh ở 5). Hàm in `lo`, `hi`, `m1`, `m2` và quyết định ở từng bước.

In [19]:
def trace_ternary(arr):
    lo, hi = 0, len(arr) - 1
    print("arr =", arr)
    step = 0
    while hi - lo > 2:
        step += 1
        third = (hi - lo) // 3
        m1, m2 = lo + third, hi - third
        a, b = arr[m1], arr[m2]
        if a < b:
            act = f"arr[m1] < arr[m2] -> lo = {m1 + 1}"
            lo = m1 + 1
        elif a > b:
            act = f"arr[m1] > arr[m2] -> hi = {m2 - 1}"
            hi = m2 - 1
        else:
            act = f"bằng nhau (hai phía đỉnh) -> lo = {m1 + 1}, hi = {m2 - 1}"
            lo, hi = m1 + 1, m2 - 1
        print(f"  vòng {step}: [{lo if a >= b else m1}, ...] m1={m1} ({a}), m2={m2} ({b}): {act}")
    cand = list(range(lo, hi + 1))
    best = max(cand, key=lambda i: arr[i])
    print(f"  còn {cand}: lấy max -> {best}")
    return best

assert trace_ternary([1, 4, 7, 10, 14, 20, 19, 15, 11, 8, 3, 0]) == 5

arr = [1, 4, 7, 10, 14, 20, 19, 15, 11, 8, 3, 0]
  vòng 1: [3, ...] m1=3 (10), m2=8 (11): arr[m1] < arr[m2] -> lo = 4
  vòng 2: [4, ...] m1=6 (19), m2=9 (8): arr[m1] > arr[m2] -> hi = 8
  vòng 3: [4, ...] m1=5 (20), m2=7 (15): arr[m1] > arr[m2] -> hi = 6
  còn [4, 5, 6]: lấy max -> 5


### 8.3. Số lần đọc: nhị phân so với tam phân và tuyến tính

Với một mảng ngọn núi dài `n`, đếm số lần `arr[i]` được đọc của từng cách:

In [20]:
def reads_of(fn, arr):
    c = Counted(arr)
    fn(c)
    return c.reads

print(f"{'n':>7} {'tuyến tính':>11} {'nhị phân [1,n-2]':>17} {'nhị phân [0,n-1]':>17} {'tam phân':>9} {'bisect':>7}")
for n in (10, 100, 1000, 10000, 100000):
    arr = mountain(n, n // 2, random.Random(n))
    row = [reads_of(peak_linear, arr), reads_of(peak_binary_inner, arr), reads_of(peak_binary_full, arr), reads_of(peak_ternary, arr), reads_of(peak_bisect, arr)]
    print(f"{n:>7} {row[0]:>11} {row[1]:>17} {row[2]:>17} {row[3]:>9} {row[4]:>7}")

      n  tuyến tính  nhị phân [1,n-2]  nhị phân [0,n-1]  tam phân  bisect
     10          12                 6                 8        12       8
    100         102                14                14        29      14
   1000        1002                20                20        42      20
  10000       10002                28                28        60      28


 100000      100002                34                34        77      34


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (nhị phân trên `[1, n - 2]`): ngắn gọn, `O(log n)`, không đọc vượt biên.

```python
class Solution:
    def peakIndexInMountainArray(self, arr: List[int]) -> int:
        left, right = 1, len(arr) - 2
        while left < right:
            mid = (left + right) >> 1
            if arr[mid] > arr[mid + 1]:
                right = mid
            else:
                left = mid + 1
        return left
```

In [21]:
class Solution:
    def peakIndexInMountainArray(self, arr: List[int]) -> int:
        left, right = 1, len(arr) - 2
        while left < right:
            mid = (left + right) >> 1
            if arr[mid] > arr[mid + 1]:
                right = mid
            else:
                left = mid + 1
        return left

sol = Solution()
for arr, expected in examples:
    got = sol.peakIndexInMountainArray(arr)
    print(arr, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[0, 1, 0] -> 1 OK
[0, 2, 1, 0] -> 1 OK
[0, 10, 5, 2] -> 1 OK


## 10. Vì sao đúng

**(a) Tính đơn điệu của vị từ.** Với ngọn núi có đỉnh `p`: `arr[i] < arr[i + 1]` với `i < p` (sườn tăng ngặt) và `arr[i] > arr[i + 1]` với `i >= p` (sườn giảm ngặt; tại `i = p` là `arr[p] > arr[p + 1]`). Vậy `down(i) = (arr[i] > arr[i + 1])` là `False` với `i < p` và `True` với `i >= p`.

**(b) Bất biến chặt nhị phân.** Ở Cách 2 và 3, ta duy trì "`p ∈ [lo, hi]`". Nếu `down(mid)` đúng thì `mid >= p`, nên `p ∈ [lo, mid]` và đặt `hi = mid` giữ bất biến. Nếu sai thì `mid < p`, nên `p ∈ [mid + 1, hi]`. Vùng `[lo, hi]` luôn nhỏ đi (`mid < hi` nên `hi = mid` hoặc `lo = mid + 1` giảm độ dài) nên vòng lặp dừng; khi `lo == hi` thì `p = lo`. Cách 2 khởi tạo `[1, n - 2]` hợp lệ vì `1 <= p <= n - 2`.

**(c) Không vượt biên.** Vì `mid = (lo + hi) >> 1 < hi <= n - 1` (hoặc `n - 2` ở Cách 2) nên `mid + 1 <= n - 1`: đọc `arr[mid + 1]` an toàn.

**(d) Cách 4 (tam phân).** Hàm tăng ngặt rồi giảm ngặt. Nếu `arr[m1] < arr[m2]` thì `m1` không thể ở phía phải đỉnh hay là đỉnh (nếu `m1 >= p` thì `m2 > m1 >= p` nên `arr[m1] > arr[m2]`), vậy `m1 < p`, loại `[lo, m1]` an toàn. Đối xứng khi `arr[m1] > arr[m2]`: `m2 > p`. Khi bằng nhau, hai điểm ở hai phía khác nhau của đỉnh (mảng ngặt không có hai điểm cùng phía bằng nhau), nên `m1 < p < m2` và loại hai đầu. Còn `<= 3` phần tử thì lấy max.

**(e) Cách 5.** `bisect_left` tìm vị trí `True` đầu tiên trong dãy `key` không giảm: đó là `p` theo (a).

**Kiểm chứng bằng máy:** so mọi cách với oracle `argmax` trên **mọi** mảng ngọn núi có giá trị trong `0..6` và độ dài 3..7 (hàng chục nghìn mảng), và trên nhiều mảng ngẫu nhiên lớn.

In [22]:
solutions = {
    "linear": peak_linear,
    "argmax": peak_argmax,
    "inner": peak_binary_inner,
    "full": peak_binary_full,
    "ternary": peak_ternary,
    "bisect": peak_bisect,
    "sol_inner": sol_inner,
    "sol_full": sol_full,
    "sol_ternary": sol_ternary,
    "sol_bisect": sol_bisect,
    "final": Solution().peakIndexInMountainArray,
}

checked = 0
for n in range(3, 8):
    for arr in product(range(7), repeat=n) if n <= 6 else []:
        arr = list(arr)
        if not is_mountain(arr):
            continue
        ref = peak_argmax(arr)
        for name, fn in solutions.items():
            assert fn(arr) == ref, (name, arr, fn(arr), ref)
        checked += 1
# n = 7, 8: sinh mọi ngọn núi bằng cách chọn tập giá trị (tránh duyệt toàn bộ tích Descartes)
for n in (7, 8):
    vals = range(9)
    for peak in range(1, n - 1):
        for left in combinations(vals, peak):                       # sườn trái tăng ngặt
            top_min = left[-1] + 1
            for top in range(top_min, 9):
                pool = [v for v in vals if v < top]
                for right in combinations(pool, n - peak - 1):      # sườn phải: chọn giá trị < top, xếp giảm dần
                    arr = list(left) + [top] + sorted(right, reverse=True)
                    assert is_mountain(arr) and peak_argmax(arr) == peak
                    for name, fn in solutions.items():
                        assert fn(arr) == peak, (name, arr)
                    checked += 1
print(f"mọi ngọn núi giá trị <= 6 (n <= 6) và <= 8 (n = 7, 8): {checked} mảng x {len(solutions)} cài đặt khớp oracle")

mọi ngọn núi giá trị <= 6 (n <= 6) và <= 8 (n = 7, 8): 30158 mảng x 11 cài đặt khớp oracle


## 11. Time / Space Complexity

Gọi `n = len(arr)` (`<= 10^5`).

| Cách | Time | Space | Số lần đọc (xấp xỉ) |
|---|---|---|---|
| 1. Quét tuyến tính | `O(n)` | `O(1)` | `~p` (tới `n`) |
| 2. Nhị phân `[1, n-2]` | `O(log n)` | `O(1)` | `~2 log2 n` |
| 3. Nhị phân `[0, n-1]` | `O(log n)` | `O(1)` | `~2 log2 n` |
| 4. Tam phân | `O(log n)` (đáy `3/2`) | `O(1)` | `~2 log_{3/2} n` |
| 5. `bisect_left` + `key` | `O(log n)` | `O(1)` | `~2 log2 n` |

Với `n = 10^5`: `log2 n ≈ 17`: chỉ khoảng 34 lần đọc so với 100.000. Đo thời gian chạy:

In [23]:
rng = random.Random(852)
tests = [mountain(100000, rng.randint(1, 99998), rng) for _ in range(30)]
tests.append(mountain(100000, 1, rng))          # đỉnh sát đầu
tests.append(mountain(100000, 99998, rng))      # đỉnh sát cuối
ref = [peak_argmax(a) for a in tests]
for name in ["linear", "inner", "full", "ternary", "bisect"]:
    t0 = time.perf_counter()
    res = [solutions[name](a) for a in tests]
    dt = (time.perf_counter() - t0) * 1000
    assert res == ref, name
    print(f"{name:8} {len(tests)} mảng n=10^5: {dt:9.3f} ms")

linear   32 mảng n=10^5:   179.133 ms
inner    32 mảng n=10^5:     0.228 ms
full     32 mảng n=10^5:     0.096 ms
ternary  32 mảng n=10^5:     0.391 ms
bisect   32 mảng n=10^5:     0.084 ms


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai, đối chiếu mọi cách với oracle `argmax`, kiểm tra **số lần đọc** (bằng chứng `O(log n)`), và test ngẫu nhiên có seed (đúng ràng buộc: mảng ngọn núi, `3 <= n <= 10^5`, giá trị `0..10^6`).

In [24]:
# 12.1 ví dụ đề
for arr, expected in examples:
    for name, fn in solutions.items():
        assert fn(arr) == expected, (name, arr)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([0, 1, 0], 1),                                   # nhỏ nhất: n = 3
    ([1, 3, 2], 1),
    ([0, 2, 1, 0], 1),
    ([0, 10, 5, 2], 1),                               # đỉnh sát đầu
    ([1, 2, 3, 4, 5, 4], 4),                          # đỉnh sát cuối
    ([1, 2, 3, 4, 3, 2, 1], 3),                       # đỉnh ở giữa, đối xứng
    ([0, 5, 6, 7, 8, 9, 1, 0], 5),
    ([3, 4, 5, 1], 2),
    ([24, 69, 100, 99, 79, 78, 67, 36, 26, 19], 2),   # ví dụ thường gặp
    (list(range(0, 50000)) + list(range(50000, 0, -1)), 50000),     # n = 100000, đỉnh ở giữa
    (list(range(0, 99999)) + [0], 99998),             # đỉnh sát cuối, n = 100000 (giá trị 0..99998 rồi 0)
    ([0] + list(range(1000000, 1000000 - 99999, -1)), 1),            # đỉnh sát đầu, giá trị lớn nhất 10^6
]
for arr, expected in edge:
    assert is_mountain(arr), ("không phải ngọn núi", arr[:5])
    for name, fn in solutions.items():
        assert fn(arr) == expected, (name, arr[:6], fn(arr), expected)
print("12.2 edge case: OK", len(edge), "ca")

# 12.3 bằng chứng O(log n): số lần đọc của nhị phân <= 2 * (log2 n + 2)
import math
rng = random.Random(1)
for n in (3, 10, 1000, 100000):
    for peak in sorted({1, n // 2, n - 2}):
        if not (0 < peak < n - 1):
            continue
        arr = mountain(n, peak, rng) if n > 3 else [0, 1, 0]
        for name in ("inner", "full", "bisect"):
            c = Counted(arr)
            assert solutions[name](c) == peak
            assert c.reads <= 2 * (math.log2(n) + 2), (name, n, c.reads)
print("12.3 số lần đọc của nhị phân <= 2 (log2 n + 2): OK")

12.1 ví dụ đề: OK
12.2 edge case: OK 12 ca


12.3 số lần đọc của nhị phân <= 2 (log2 n + 2): OK


In [25]:
# 12.4 phản ví dụ: các cách sai
def wrong_le_hi_mid(arr):               # while lo <= hi kèm hi = mid -> có thể lặp vô hạn (ta chặn bằng số vòng)
    lo, hi = 0, len(arr) - 1
    for _ in range(1000):
        if lo > hi:
            break
        mid = (lo + hi) >> 1
        if arr[mid] > arr[mid + 1] if mid + 1 < len(arr) else True:
            hi = mid
        else:
            lo = mid + 1
    else:
        return "LẶP VÔ HẠN"
    return lo

def wrong_out_of_range(arr):            # while lo <= hi và đọc arr[mid + 1]: vượt biên khi đỉnh ở cuối mảng (bài 162)
    lo, hi = 0, len(arr) - 1
    while lo <= hi:
        mid = (lo + hi) >> 1
        if arr[mid] < arr[mid + 1]:     # có thể vượt biên khi mid = n - 1
            lo = mid + 1
        else:
            hi = mid - 1
    return lo

def wrong_direction(arr):               # nhầm hướng
    lo, hi = 1, len(arr) - 2
    while lo < hi:
        mid = (lo + hi) >> 1
        if arr[mid] > arr[mid + 1]:
            lo = mid + 1                # SAI: đang xuống mà lại đi sang phải
        else:
            hi = mid
    return lo

def wrong_value(arr):                   # trả giá trị đỉnh thay vì chỉ số
    return arr[Solution().peakIndexInMountainArray(arr)]

def wrong_prev_compare(arr):            # so với arr[mid - 1] nhưng thu hẹp sai
    lo, hi = 1, len(arr) - 2
    while lo < hi:
        mid = (lo + hi) >> 1
        if arr[mid - 1] < arr[mid]:
            hi = mid - 1                # SAI: đang lên thì đỉnh ở bên phải
        else:
            lo = mid + 1
    return lo

wrongs = [
    ("while lo <= hi + hi = mid", wrong_le_hi_mid, [0, 1, 2, 3, 2, 1]),
    ("đọc vượt biên", wrong_out_of_range, [1, 2, 3]),
    ("nhầm hướng", wrong_direction, [0, 2, 5, 9, 8, 3, 1]),
    ("trả giá trị thay vì chỉ số", wrong_value, [0, 2, 5, 9, 8, 3, 1]),
    ("so arr[mid-1] rồi thu hẹp sai", wrong_prev_compare, [0, 2, 5, 9, 8, 3, 1]),
]
for name, fn, arr in wrongs:
    truth = peak_argmax(arr)
    try:
        got = fn(arr)
    except IndexError as ex:
        got = f"IndexError ({ex})"
    print(f"{name:34} arr={arr}: sai = {got}, đúng = {truth}")
    assert got != truth, name
print("12.4 phản ví dụ: OK (các cách sai đều bị lộ)")

while lo <= hi + hi = mid          arr=[0, 1, 2, 3, 2, 1]: sai = LẶP VÔ HẠN, đúng = 3
đọc vượt biên                      arr=[1, 2, 3]: sai = IndexError (list index out of range), đúng = 2
nhầm hướng                         arr=[0, 2, 5, 9, 8, 3, 1]: sai = 5, đúng = 3
trả giá trị thay vì chỉ số         arr=[0, 2, 5, 9, 8, 3, 1]: sai = 9, đúng = 3
so arr[mid-1] rồi thu hẹp sai      arr=[0, 2, 5, 9, 8, 3, 1]: sai = 1, đúng = 3
12.4 phản ví dụ: OK (các cách sai đều bị lộ)


In [26]:
# 12.5 ngẫu nhiên có seed, đúng ràng buộc
rng = random.Random(2852)
for _ in range(1500):
    n = rng.randint(3, rng.choice([5, 20, 300]))
    peak = rng.randint(1, n - 2)
    arr = mountain(n, peak, rng)
    assert is_mountain(arr) and peak_argmax(arr) == peak
    for name, fn in solutions.items():
        assert fn(arr) == peak, (name, n, peak)
print("12.5 ngẫu nhiên nhỏ/vừa: OK")

rng = random.Random(77)
for _ in range(40):
    n = rng.randint(50000, 100000)
    peak = rng.randint(1, n - 2)
    arr = mountain(n, peak, rng)
    res = {name: fn(arr) for name, fn in solutions.items() if name not in ("linear", "argmax")}
    assert set(res.values()) == {peak}, (n, peak, res)
print("12.6 ngẫu nhiên lớn (n tới 10^5): OK")

12.5 ngẫu nhiên nhỏ/vừa: OK


12.6 ngẫu nhiên lớn (n tới 10^5): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — `while lo <= hi` kèm `hi = mid` (lặp vô hạn)

Khi `lo == hi` và nhánh `hi = mid` được chọn, `hi` không đổi, vòng lặp không bao giờ kết thúc.

In [27]:
arr = [0, 1, 2, 3, 2, 1]
print("sai:", wrong_le_hi_mid(arr), "| đúng:", Solution().peakIndexInMountainArray(arr))
assert wrong_le_hi_mid(arr) == "LẶP VÔ HẠN"

sai: LẶP VÔ HẠN | đúng: 3


Vì sao sai: với `hi = mid` ta phải dùng `while lo < hi` (khoảng thu nhỏ chắc chắn), vì `hi = mid` không loại bỏ `mid`. **Đúng:** khuôn `while lo < hi` ở mục 3.3.

### Lỗi 2 — Đọc `arr[mid + 1]` vượt biên

Với khuôn `while lo <= hi` và `hi = n - 1`, `mid` có thể bằng `n - 1`, khi đó `arr[mid + 1]` không tồn tại. Ở bài này đề đảm bảo đỉnh không ở cuối nên lỗi không lộ ra; nhưng với mảng **chỉ tăng** (ví dụ bài Find Peak Element) nó nổ.

In [28]:
arr = [1, 2, 3]            # không phải ngọn núi (đỉnh ở cuối) - hợp lệ cho bài 162
try:
    print(wrong_out_of_range(arr))
except IndexError as ex:
    print("lỗi:", type(ex).__name__, "-", ex)
print("đúng (Cách 3, khuôn while lo < hi):", peak_binary_full(arr))
assert peak_binary_full(arr) == 2

lỗi: IndexError - list index out of range
đúng (Cách 3, khuôn while lo < hi): 2


Vì sao sai: không bảo đảm `mid + 1` hợp lệ. **Đúng:** `while lo < hi` (suy ra `mid < hi`, nên `mid + 1 <= hi <= n - 1`) hoặc khởi tạo `hi = n - 2`.

### Lỗi 3 — Nhầm hướng thu hẹp

Khi đang **xuống** (`arr[mid] > arr[mid + 1]`) đỉnh ở `mid` hoặc bên **trái**; đi sang phải là sai.

In [29]:
arr = [0, 2, 5, 9, 8, 3, 1]
print("sai:", wrong_direction(arr), "| đúng:", Solution().peakIndexInMountainArray(arr))
assert wrong_direction(arr) != 3

sai: 5 | đúng: 3


Vì sao sai: sau `mid = 3` (`9 > 8`) lẽ ra `hi = 3`, nhưng `lo = 4` bỏ mất đỉnh. **Đúng:** xuống -> `hi = mid`, lên -> `lo = mid + 1`.

### Lỗi 4 — Trả về giá trị đỉnh thay vì chỉ số

Đề hỏi **chỉ số** của đỉnh.

In [30]:
arr = [0, 2, 5, 9, 8, 3, 1]
print("sai (giá trị):", wrong_value(arr), "| đúng (chỉ số):", Solution().peakIndexInMountainArray(arr))
assert wrong_value(arr) == 9 and Solution().peakIndexInMountainArray(arr) == 3

sai (giá trị): 9 | đúng (chỉ số): 3


Vì sao sai: `arr[3] = 9` là giá trị; chỉ số là `3`. **Đúng:** `return left` (chỉ số).

### Lỗi 5 — So với `arr[mid - 1]` rồi thu hẹp sai

So với phần tử **bên trái** cũng được, nhưng phải đảo đúng hướng: `arr[mid - 1] < arr[mid]` (đang lên) -> đỉnh ở `mid` hoặc bên phải.

In [31]:
arr = [0, 2, 5, 9, 8, 3, 1]
print("sai:", wrong_prev_compare(arr), "| đúng:", Solution().peakIndexInMountainArray(arr))
assert wrong_prev_compare(arr) != 3

sai: 1 | đúng: 3


Vì sao sai: đang lên mà đặt `hi = mid - 1` bỏ mất đỉnh. **Đúng:** lên -> `lo = mid` (cẩn thận vòng lặp vô hạn nếu `lo = mid`; thường dùng so với `arr[mid + 1]` như Cách 2).

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "tìm đỉnh/đáy/điểm chuyển" trong dãy **đơn đỉnh** và yêu cầu `O(log n)` -> chặt nhị phân trên **độ dốc** (`arr[mid]` so với `arr[mid + 1]`).
- **Chặt nhị phân không cần mảng sắp xếp:** chỉ cần vị từ "sai ... sai, đúng ... đúng".
- **Khuôn `while lo < hi`, `hi = mid` / `lo = mid + 1`** an toàn: không vượt biên, luôn kết thúc.
- **Tam phân** cho hàm đơn đỉnh tổng quát (kể cả số thực), nhưng nhị phân trên độ dốc gọn và nhanh hơn khi so sánh kề nhau.
- `bisect_left(..., key=...)` (Python 3.10+) rút gọn code.
- Cùng họ: Find Peak Element (162), Find in Mountain Array (1095), Valid Mountain Array (941), Longest Mountain in Array (845).
- Test nên có: `n = 3`, đỉnh sát hai đầu, đỉnh ở giữa, `n = 10^5`.

## 15. Biến thể

1. **Find Peak Element (162):** mảng bất kỳ (hai phần tử kề khác nhau), tìm **một** đỉnh cục bộ: cùng khuôn nhị phân; có **nhiều đáp án đúng** nên kiểm tra bằng hàm hợp lệ.
2. **Valid Mountain Array (941):** kiểm tra một mảng có phải ngọn núi không (hai con trỏ).
3. **Find in Mountain Array (1095):** tìm một giá trị trong ngọn núi: tìm đỉnh rồi nhị phân hai sườn.
4. **Số lần đọc tối thiểu** (interactive): bài gốc của 1095 giới hạn số lần gọi `get`.

Dưới đây là code cho biến thể 1 (kiểm tra bằng hàm hợp lệ) và 3.

In [32]:
def find_peak_element(nums: List[int]) -> int:
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) >> 1
        if nums[mid] < nums[mid + 1]:
            lo = mid + 1
        else:
            hi = mid
    return lo

def is_local_peak(nums, i) -> bool:
    left = nums[i - 1] if i > 0 else float("-inf")
    right = nums[i + 1] if i + 1 < len(nums) else float("-inf")
    return nums[i] > left and nums[i] > right

rng = random.Random(162)
for _ in range(1000):
    n = rng.randint(1, 30)
    nums = [rng.randint(0, 10)]
    while len(nums) < n:
        v = rng.randint(0, 10)
        if v != nums[-1]:                                   # hai phần tử kề khác nhau (ràng buộc của 162)
            nums.append(v)
    i = find_peak_element(nums)
    assert is_local_peak(nums, i), (nums, i)               # nhiều đáp án đúng -> kiểm tra tính hợp lệ

def search_in_mountain(arr: List[int], target: int) -> int:
    p = peak_binary_inner(arr)
    lo, hi = 0, p
    while lo < hi:                                          # sườn tăng: vị trí đầu tiên >= target
        mid = (lo + hi) >> 1
        if arr[mid] >= target:
            hi = mid
        else:
            lo = mid + 1
    if arr[lo] == target:
        return lo
    lo, hi = p, len(arr) - 1
    while lo < hi:                                          # sườn giảm: vị trí đầu tiên <= target
        mid = (lo + hi) >> 1
        if arr[mid] <= target:
            hi = mid
        else:
            lo = mid + 1
    return lo if arr[lo] == target else -1

for _ in range(300):
    n = rng.randint(3, 40)
    arr = mountain(n, rng.randint(1, n - 2), rng)
    for target in rng.sample(arr, min(3, n)) + [rng.randint(0, 10 ** 6)]:
        got = search_in_mountain(arr, target)
        expected = arr.index(target) if target in arr else -1
        assert got == expected, (arr, target, got, expected)
print("Biến thể: OK")

Biến thể: OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét tuyến tính | Dừng ở `arr[i] > arr[i+1]` đầu tiên | `O(n)` | `O(1)` | Oracle; chưa cần `O(log n)` |
| 2. Nhị phân `[1, n-2]` | Vị từ độ dốc | `O(log n)` | `O(1)` | **Cách nộp chính** |
| 3. Nhị phân `[0, n-1]` | Cùng ý, biên rộng hơn | `O(log n)` | `O(1)` | Tổng quát hơn (đỉnh ở hai đầu) |
| 4. Tam phân | So hai điểm, loại 1/3 | `O(log n)` | `O(1)` | Hàm đơn đỉnh không có "độ dốc" kề nhau |
| 5. `bisect_left` + `key` | Thư viện | `O(log n)` | `O(1)` | Code ngắn nhất |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng sau, rồi chạy để kiểm tra:

```python
arr = [1, 3, 5, 4, 2]
print([arr[i] > arr[i + 1] for i in range(len(arr) - 1)])
print(Solution().peakIndexInMountainArray(arr))
print((3 + 7) >> 1, (4 + 5) >> 1)
```

In [33]:
# Bài 1: ghi dự đoán của bạn (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `arr = [1, 4, 7, 10, 12, 11, 6, 2]` (n = 8): chạy Cách 2 bằng tay; ghi danh sách các cặp `(left, right, mid)` qua từng vòng và đáp án.

In [34]:
# Bài 2: điền kết quả tính tay
hand_steps = None     # list các tuple (left, right, mid)
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành nhị phân trên `[1, n - 2]`.

In [35]:
def exercise_3_binary(arr: List[int]) -> int:
    left, right = 1, len(arr) - 2
    # TODO 1: lặp khi left < right
    #     TODO 2: mid = (left + right) >> 1
    #     TODO 3: nếu arr[mid] > arr[mid + 1] (đang xuống) thì right = mid
    #     TODO 4: ngược lại left = mid + 1
    # TODO 5: trả về left
    return None


# assert exercise_3_binary([0, 1, 0]) == 1
# assert exercise_3_binary([0, 2, 5, 9, 8, 3, 1]) == 3
# assert exercise_3_binary([1, 2, 3, 4, 5, 4]) == 4

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, hãy tự viết `exercise_4_ternary(arr)` bằng tam phân (so `arr[m1]` với `arr[m2]`).

In [36]:
def exercise_4_ternary(arr: List[int]) -> int:
    # TODO
    pass


# assert exercise_4_ternary([0, 1, 0]) == 1
# assert exercise_4_ternary([0, 2, 5, 9, 8, 3, 1]) == 3
# assert exercise_4_ternary([1, 2, 3, 4, 5, 4]) == 4

### Bài 5 — Mini challenge: Find Peak Element (khó)

Cho mảng `nums` mà hai phần tử kề nhau luôn khác nhau (không nhất thiết là ngọn núi). Trả về chỉ số của **một** đỉnh cục bộ (phần tử lớn hơn cả hai láng giềng; biên coi láng giềng ngoài mảng là `-vô cùng`) trong `O(log n)`. Có thể có nhiều đáp án đúng; kiểm tra bằng `is_local_peak`.

```python
exercise_5_find_peak([1, 2, 3, 1])         -> 2
exercise_5_find_peak([1, 2, 1, 3, 5, 6, 4]) -> 1 hoặc 5
exercise_5_find_peak([5])                  -> 0
```
Gợi ý: cùng khuôn `while lo < hi`, so `nums[mid]` với `nums[mid + 1]`.

In [37]:
def exercise_5_find_peak(nums: List[int]) -> int:
    # TODO
    pass


# for nums in ([1, 2, 3, 1], [1, 2, 1, 3, 5, 6, 4], [5], [3, 2, 1], [1, 2, 3]):
#     assert is_local_peak(nums, exercise_5_find_peak(nums)), nums

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `arr[i] > arr[i + 1]` kiểm tra "đang xuống" tại `i`.
- Mức 2: `[1, 3, 5, 4, 2]`: lên, lên, xuống, xuống; đỉnh ở chỉ số 2.
- Mức 3: `[False, False, True, True]`, `2`, `5 4`.

**Bài 2**
- Mức 1: `left = 1`, `right = 6`, `mid = 3`.
- Mức 2: `arr[3] = 10 < arr[4] = 12` -> `left = 4`; `mid = 5`: `11 > 6` -> `right = 5`.
- Mức 3: `[(1, 6, 3), (4, 6, 5), (4, 5, 4)]`, đáp án `4` (`arr[4] = 12`).

**Bài 3**
- Mức 1: `while left < right:`.
- Mức 2: `mid = (left + right) >> 1`.
- Mức 3: `if arr[mid] > arr[mid + 1]: right = mid` / `else: left = mid + 1`; `return left`.

**Bài 4**
- Mức 1: `lo, hi = 0, n - 1`; lặp khi `hi - lo > 2`.
- Mức 2: `third = (hi - lo) // 3`; `m1 = lo + third`, `m2 = hi - third`.
- Mức 3: `<` -> `lo = m1 + 1`; `>` -> `hi = m2 - 1`; bằng -> cả hai; cuối lấy `max(range(lo, hi + 1), key=lambda i: arr[i])`.

**Bài 5**
- Mức 1: nếu `nums[mid] < nums[mid + 1]` thì có đỉnh bên phải `mid`.
- Mức 2: ngược lại có đỉnh ở `mid` hoặc bên trái.
- Mức 3: `lo, hi = 0, n - 1`; `while lo < hi: mid...`; `return lo`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [38]:
arr = [1, 3, 5, 4, 2]
print([arr[i] > arr[i + 1] for i in range(len(arr) - 1)])
print(Solution().peakIndexInMountainArray(arr))
print((3 + 7) >> 1, (4 + 5) >> 1)

assert [arr[i] > arr[i + 1] for i in range(len(arr) - 1)] == [False, False, True, True]
assert Solution().peakIndexInMountainArray(arr) == 2
assert ((3 + 7) >> 1, (4 + 5) >> 1) == (5, 4)

[False, False, True, True]
2
5 4


### Đáp án Bài 2

`arr = [1, 4, 7, 10, 12, 11, 6, 2]`: `(1, 6, 3)` (`10 < 12` -> `left = 4`), `(4, 6, 5)` (`11 > 6` -> `right = 5`), `(4, 5, 4)` (`12 > 11` -> `right = 4`), dừng với `left = right = 4`. Đáp án `4`.

In [39]:
arr = [1, 4, 7, 10, 12, 11, 6, 2]
steps = []
left, right = 1, len(arr) - 2
while left < right:
    mid = (left + right) >> 1
    steps.append((left, right, mid))
    if arr[mid] > arr[mid + 1]:
        right = mid
    else:
        left = mid + 1
hand_steps_solution = [(1, 6, 3), (4, 6, 5), (4, 5, 4)]
hand_answer_solution = 4
assert steps == hand_steps_solution and left == hand_answer_solution == peak_argmax(arr)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [40]:
def exercise_3_binary_solution(arr: List[int]) -> int:
    left, right = 1, len(arr) - 2
    while left < right:
        mid = (left + right) >> 1
        if arr[mid] > arr[mid + 1]:
            right = mid
        else:
            left = mid + 1
    return left

assert exercise_3_binary_solution([0, 1, 0]) == 1
assert exercise_3_binary_solution([0, 2, 5, 9, 8, 3, 1]) == 3
assert exercise_3_binary_solution([1, 2, 3, 4, 5, 4]) == 4
rng = random.Random(3)
for _ in range(300):
    n = rng.randint(3, 60)
    peak = rng.randint(1, n - 2)
    arr = mountain(n, peak, rng)
    assert exercise_3_binary_solution(arr) == peak
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [41]:
def exercise_4_ternary_solution(arr: List[int]) -> int:
    lo, hi = 0, len(arr) - 1
    while hi - lo > 2:
        third = (hi - lo) // 3
        m1, m2 = lo + third, hi - third
        if arr[m1] < arr[m2]:
            lo = m1 + 1
        elif arr[m1] > arr[m2]:
            hi = m2 - 1
        else:
            lo, hi = m1 + 1, m2 - 1
    return max(range(lo, hi + 1), key=lambda i: arr[i])

assert exercise_4_ternary_solution([0, 1, 0]) == 1
assert exercise_4_ternary_solution([0, 2, 5, 9, 8, 3, 1]) == 3
assert exercise_4_ternary_solution([1, 2, 3, 4, 5, 4]) == 4
rng = random.Random(4)
for _ in range(300):
    n = rng.randint(3, 80)
    peak = rng.randint(1, n - 2)
    arr = mountain(n, peak, rng)
    assert exercise_4_ternary_solution(arr) == peak
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [42]:
def exercise_5_find_peak_solution(nums: List[int]) -> int:
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) >> 1
        if nums[mid] < nums[mid + 1]:
            lo = mid + 1
        else:
            hi = mid
    return lo

for nums in ([1, 2, 3, 1], [1, 2, 1, 3, 5, 6, 4], [5], [3, 2, 1], [1, 2, 3]):
    assert is_local_peak(nums, exercise_5_find_peak_solution(nums)), nums
rng = random.Random(5)
for _ in range(500):
    n = rng.randint(1, 40)
    nums = [rng.randint(0, 20)]
    while len(nums) < n:
        v = rng.randint(0, 20)
        if v != nums[-1]:
            nums.append(v)
    assert is_local_peak(nums, exercise_5_find_peak_solution(nums)), nums
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (nhị phân trên độ dốc)

```python
class Solution:
    def peakIndexInMountainArray(self, arr: List[int]) -> int:
        left, right = 1, len(arr) - 2            # đỉnh không ở hai đầu
        while left < right:
            mid = (left + right) >> 1
            if arr[mid] > arr[mid + 1]:          # đang xuống: đỉnh ở mid hoặc bên trái
                right = mid
            else:                                # đang lên: đỉnh ở bên phải mid
                left = mid + 1
        return left
```

Công thức nhớ nhanh: **xuống -> `right = mid`, lên -> `left = mid + 1`**, khuôn `while left < right`, trả `left`.

### Checklist trước khi submit

- [ ] Dùng khuôn `while left < right` (không `<=`) khi có `right = mid`?
- [ ] `arr[mid + 1]` luôn hợp lệ (`mid < right <= n - 2` hoặc `n - 1`)?
- [ ] Đang **xuống** (`arr[mid] > arr[mid + 1]`) thì `right = mid`; đang **lên** thì `left = mid + 1`?
- [ ] Trả về **chỉ số**, không phải giá trị?
- [ ] Không dùng quét tuyến tính khi đề đòi `O(log n)`?
- [ ] Đã thử: `n = 3`, đỉnh sát đầu, đỉnh sát cuối, `n = 10^5`?
- [ ] Tên method đúng `peakIndexInMountainArray(self, arr)` và trả `int`?